# Classification with Naive Bayes

In [13]:
import pandas as pd

## The watermelon dataset

In the following we use the Watermelon dataset from Zhou's textbook, where each record represents a watermelon with its features (color, sound, texture, etc) and with a boolean value indicating if the watermelon is ripe or not.
Notice:
Input features have different types: *color, root, sound, texture, umbilicus, surface* are categorical (have a finite number of nominal values that can only enumerated and compared by equality operators), while *density, sugar* are numerical (continuous) and mathematical operators can be used to transform their values.

We can easily read this CSV dataset using pandas.

In [14]:
dataset = pd.read_csv('datasets/watermelon-dataset.csv')
dataset

,ID,color,root,sound,texture,umbilicus,surface,density,sugar,ripe
0,1,green,curly,muffled,clear,hollow,hard,0.697,0.460,True
1,2,dark,curly,dull,clear,hollow,hard,0.774,0.376,True
2,3,dark,curly,muffled,clear,hollow,hard,0.634,0.264,True
3,4,green,curly,dull,clear,hollow,hard,0.608,0.318,True
4,5,light,curly,muffled,clear,hollow,hard,0.556,0.215,True
5,6,green,slightly curly,muffled,clear,slightly hollow,soft,0.403,0.237,True
6,7,dark,slightly curly,muffled,slightly blurry,slightly hollow,soft,0.481,0.149,True
7,8,dark,slightly curly,muffled,clear,slightly hollow,hard,0.437,0.211,True
8,9,dark,slightly curly,dull,slightly blurry,slightly hollow,hard,0.666,0.091,False
9,10,green,straight,crisp,clear,flat,soft,0.243,0.267,False


# Your work: what you have to do

- 1. Use `CategoricalNB` and `GaussianNB` from `scikit-learn` to build a Naive Bayes classifier on the watermelon dataset. For the estimated probabilities, consider the Laplace correction techniques.

- 2. Using the previously trained model, predict the ripe/unripe class of the test example proposed by the book:
*ID=T1, color=green, root=curly, sound=muffled, texture=clear, umbilicous=hollow, surface=hard, density=0.697, sugar=0.460. *

- 3. Giving the dataset size, consider now an hold-out technique in which you draw randomly a single test example and you train the model on the remaining examples.

- 4. Implement statistical tests (e.g. `ttest_rel` in [scipy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html)) to compare model trained at 3. to a Decision Tree on the same dataset with an hold-out method. Have the observed differences in predictive performance statistical  significance?

## Parte laboratorio fatta da meo su hold-out ##

#### Traduzione parte meo su moodle #### 

- Predirremo la classe di un esempio di test T1:

<ID=T1, color=green, root=curly, sound=muffled, texture=clear, umbilicous=hollow, surface=hard, density=0.697, sugar=0.460> <--questo è il primo esempio del dataset ID=1

- Poi, useremo la tecnica di hold-out:

- Confronteremo le performance predittive con la matrice di confusione e le curve ROC di naive Bayes e di decision trees con la tecnica di hold-out. 

- Troveremo se le differenze osservate di accuratezza predittiva con la tecnica di hold-out, tra naive Bayes e decision tree sono statisticamente significative.

##### Preparazione dati 

In [15]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
# Separa features categoriche e numeriche
categorical_cols = ['color', 'root', 'sound', 'texture', 'umbilicus', 'surface']
numerical_cols = ['density', 'sugar']

X_categorical = dataset[categorical_cols]
X_numerical = dataset[numerical_cols]
y = dataset['ripe']

# Encoding delle features categoriche (testo → numeri)
label_encoders = {}
X_cat_encoded = X_categorical.copy()

for col in categorical_cols:
    le = LabelEncoder()
    X_cat_encoded[col] = le.fit_transform(X_categorical[col])
    label_encoders[col] = le

# Encoding del target
y_encoder = LabelEncoder()
y_encoded = y_encoder.fit_transform(y)

print(f"\nTarget encoding: {y.unique()} → {np.unique(y_encoded)}")



Target encoding: [ True False] → [0 1]


In [16]:
from sklearn.naive_bayes import CategoricalNB, GaussianNB
# Crea e addestra i classificatori con correzione di Laplace
categorical_nb = CategoricalNB(alpha=1.0, fit_prior=False)  # alpha=1.0 = correzione di Laplace
gaussian_nb = GaussianNB()

categorical_nb.fit(X_cat_encoded, y_encoded)
gaussian_nb.fit(X_numerical, y_encoded)

print(f"\nModelli addestrati:")
print(f"CategoricalNB: {X_cat_encoded.shape[1]} features categoriche")
print(f"GaussianNB: {X_numerical.shape[1]} features numeriche")
print(f"Correzione di Laplace: alpha={categorical_nb.alpha}")



Modelli addestrati:
CategoricalNB: 6 features categoriche
GaussianNB: 2 features numeriche
Correzione di Laplace: alpha=1.0


#### Combinazione probabilistica features categoriche e numeriche(gaussiane) con calcolo della accuracy

In [17]:
# Funzione per predizione combinata
def predict_combined(X_cat_encoded, X_num):
    prob_cat = categorical_nb.predict_proba(X_cat_encoded) 
    prob_num = gaussian_nb.predict_proba(X_num)
    
    # Combinazione diretta
    prob_combined = prob_cat * prob_num
    
    # Normalizza (necessario perché la moltiplicazione non preserva la normalizzazione)
    prob_combined = prob_combined / prob_combined.sum(axis=1, keepdims=True)
    
    predictions = np.argmax(prob_combined, axis=1)
    return predictions, prob_combined
# Test accuracy
predictions, probabilities = predict_combined(X_cat_encoded, X_numerical)
accuracy = np.mean(predictions == y_encoded)
print(f"\nAccuracy training: {accuracy:.10f}")


Accuracy training: 0.8235294118


## Punto 2 ##

In [ ]:
test_example = {
    'color': 'green',
    'root': 'curly',
    'sound': 'muffled',
    'texture': 'clear',
    'umbilicus': 'hollow',
    'surface': 'hard',
    'density': 0.697,
    'sugar': 0.460
}
print("Esempio T1 da predire:")


Esempio T1 da predire:
  color: green
  root: curly
  sound: muffled
  texture: clear
  umbilicus: hollow
  surface: hard
  density: 0.697
  sugar: 0.46


In [ ]:
# Prepara l'esempio per la predizione
# 1. Encoding delle features categoriche
test_categorical = []
for col in categorical_cols:
    # Converti il valore usando il LabelEncoder già addestrato
    encoded_value = label_encoders[col].transform([test_example[col]])[0]
    test_categorical.append(encoded_value)
    print(f"{col}: '{test_example[col]}' → {encoded_value}")

test_numerical = [test_example['density'], test_example['sugar']]

color: 'green' → 1
root: 'curly' → 0
sound: 'muffled' → 2
texture: 'clear' → 1
umbilicus: 'hollow' → 1
surface: 'hard' → 0

Features categoriche encoded: [np.int64(1), np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(0)]
Features numeriche: [0.697, 0.46]


In [24]:
# 3. Predizione usando i modelli addestrati - VERSIONE CORRETTA
# Crea DataFrame invece di array per mantenere i nomi delle colonne
test_cat_df = pd.DataFrame([test_categorical], columns=categorical_cols)
test_num_df = pd.DataFrame([test_numerical], columns=numerical_cols)

print(f"Test categoriche DataFrame: {test_cat_df}")
print(f"Test numeriche DataFrame: {test_num_df}")

# Probabilità da ogni modello (nessun warning!)
prob_cat = categorical_nb.predict_proba(test_cat_df)[0]
prob_num = gaussian_nb.predict_proba(test_num_df)[0]

print(f"\nProbabilità da features categoriche: {prob_cat}")
print(f"Probabilità da features numeriche: {prob_num}")

Test categoriche DataFrame:    color  root  sound  texture  umbilicus  surface
0      1     0      2        1          1        0
Test numeriche DataFrame:    density  sugar
0    0.697   0.46

Probabilità da features categoriche: [0.04991258 0.95008742]
Probabilità da features numeriche: [0.04164757 0.95835243]


In [28]:
# 4. Combinazione delle probabilità per la predizione finale
prob_combined = prob_cat * prob_num
# Normalizza le probabilità combinate perché la moltiplicazione 
# non preserva la normalizzazione, quindi sommano a 1
prob_combined = prob_combined / prob_combined.sum()

print(f"Probabilità combinate: {prob_combined}")

# 5. Predizione finale
predicted_class_idx = np.argmax(prob_combined)
predicted_class = y_encoder.inverse_transform([predicted_class_idx])[0]
confidence = prob_combined[predicted_class_idx]

print(f"\n=== RISULTATO PREDIZIONE T1 ===")
print(f"Classe predetta: {predicted_class}")
print(f"Confidenza: {confidence:.3f}")
print(f"Probabilità FALSE: {prob_combined[0]:.3f}")
print(f"Probabilità TRUE: {prob_combined[1]:.3f}")

# 6. Confronto con il valore reale
real_class = dataset.iloc[0]['ripe']
print(f"\nClasse reale ripe: {real_class}")
print(f"Predizione corretta: {'si' if predicted_class == real_class else 'no'}")

Probabilità combinate: [0.00227783 0.99772217]

=== RISULTATO PREDIZIONE T1 ===
Classe predetta: True
Confidenza: 0.998
Probabilità FALSE: 0.002
Probabilità TRUE: 0.998

Classe reale ripe: True
Predizione corretta: si
